## Make UMAP

In [ ]:
import pandas as pd

annotations = pd.read_csv(
    "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/metadata/"
    "marloes_20230622_adata_obs_celltyping.tsv",
    sep="\t",
    low_memory=False,
)

print("Columns:", annotations.columns.tolist())

for column in ["mark", "celltype_general", "celltype"]:
    print(f"\n{column}:", sorted(annotations[column].dropna().unique().tolist()))

In [ ]:
import matplotlib.pyplot as plt

def plot_umap(level_of_detail="lineage"):
    annotations = pd.read_csv(
        "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/metadata/"
        "marloes_20230622_adata_obs_celltyping.tsv",
        sep="\t",
        usecols=["mark", level_of_detail, "X_umap-0", "X_umap-1"],
    )

    with plt.style.context("tableau-colorblind10"):
        fig, axes = plt.subplots(
            1, 2, figsize=(12, 5), sharex=True, sharey=True
        )

        for ax, mark in zip(axes, ["H3K4me3", "H3K27me3"]):
            for label, cells in annotations.groupby(level_of_detail):
                cells = cells[cells["mark"] == mark]
                ax.scatter(
                    cells["X_umap-0"], cells["X_umap-1"],
                    s=2, label=label,
                )
            ax.set(
                title=mark, xlabel="UMAP1", ylabel="UMAP2", aspect="equal"
            )

        handles, labels = axes[1].get_legend_handles_labels()
        fig.legend(
            handles, labels,
            title=level_of_detail,
            loc="upper center",
            bbox_to_anchor=(0.5, 0),
            ncol=4,
            markerscale=8,
            fontsize=9,
        )

        plt.tight_layout()
        plt.show()

In [ ]:
levels_of_detail = ["lineage", "celltype_general", "celltype"]
for level_of_detail in levels_of_detail:
    plot_umap(level_of_detail)

## Check if cells match

In [ ]:
annotations = pd.read_csv(
    "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/metadata/"
    "marloes_20230622_adata_obs_celltyping.tsv",
    sep="\t",
    low_memory=False,
)

print(annotations.columns.tolist())
display(annotations.head(3).T)

In [ ]:
from pathlib import Path
import anndata as ad

data_dir = Path("/home/michiel/thesis/data/gastruloid_tchic_counts")
bin_cell_ids = {}

for path in sorted(data_dir.glob("*.h5ad")):
    adata = ad.read_h5ad(path, backed="r")
    try:
        bin_cell_ids[path.name] = adata.obs_names.copy()
    finally:
        adata.file.close()

    print(f"\n{path.name}")
    print("\n".join(bin_cell_ids[path.name][:3]))

In [ ]:
results = []

for filename, ids in bin_cell_ids.items():
    ids = ids.str.replace(r"^PZ-MB-TChIC-", "", regex=True)

    for barcode in ["bc_nla", "bc_vasa"]:
        tsv_ids = annotations["sample"] + "::" + annotations[barcode]

        results.append({
            "file": filename,
            "barcode": barcode,
            "cells": len(ids),
            "matched": ids.isin(tsv_ids).sum(),
            "bin_ids_unique": ids.is_unique,
            "tsv_ids_unique": tsv_ids.is_unique,
        })

display(pd.DataFrame(results))

In deze notebook hebben we het annotatie-TSV bekeken en de bestaande UMAP-coördinaten geplot, ingekleurd op lineage, algemeen celtype of gedetailleerd celtype. Vervolgens hebben we de cel-ID’s uit de twee binned histoonbestanden vergeleken met de TSV. Na het verwijderen van het prefix PZ-MB-TChIC- matchen alle cellen uniek via sample + "::" + bc_nla. We hebben geen nieuwe UMAP berekend en geen countwaarden geanalyseerd.

## Check counts

In [ ]:
import numpy as np

adata = ad.read_h5ad(
    data_dir / "gast_H3K4me3_2000_bins_ctype.h5ad",
    backed="r",
)

try:
    counts = adata.X[:3, :]
    counts = counts.toarray() if hasattr(counts, "toarray") else np.asarray(counts)

    # selected = np.arange(min(10000, counts.shape[1]))
    selected = np.flatnonzero((counts > 0).any(axis=0))[:100]
    cell_names = adata.obs_names[:3].tolist()
    bin_names = adata.var_names[selected].tolist()
finally:
    adata.file.close()

fig, ax = plt.subplots(figsize=(14, 3))
heatmap = ax.imshow(counts[:, selected], aspect="auto", cmap="viridis")
ax.set_yticks(range(3), labels=cell_names)
ax.set_xlabel("Genomic bins (first 100 with signal in these cells)")
ax.set_title("H3K4me3 counts")
fig.colorbar(heatmap, ax=ax, label="Count")
plt.tight_layout()
plt.show()

# Show exact counts for the first ten selected bins.
display(pd.DataFrame(
    counts[:, selected[:10]],
    index=cell_names,
    columns=bin_names[:10],
))

In [ ]:
display(pd.DataFrame({
    "total_counts": counts.sum(axis=1),
    "bins_with_signal": (counts > 0).sum(axis=1),
    "zero_bins_percent": 100 * (counts == 0).mean(axis=1),
}, index=cell_names))

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path

data_dir = Path("/home/michiel/thesis/data/gastruloid_tchic_counts")
cell_summaries, bin_summaries, metadata = [], [], []

for path in sorted(data_dir.glob("*.h5ad")):
    print(f"Reading {path.name}...")
    adata = ad.read_h5ad(path, backed="r")

    try:
        total_counts = np.zeros(adata.n_obs)
        detected_bins = np.zeros(adata.n_obs, dtype=int)
        bins_with_signal = np.zeros(adata.n_vars, dtype=bool)

        for start in range(0, adata.n_obs, 256):
            stop = min(start + 256, adata.n_obs)
            block = adata.X[start:stop, :]
            positive = block > 0

            total_counts[start:stop] = np.asarray(block.sum(axis=1)).ravel()
            detected_bins[start:stop] = np.asarray(positive.sum(axis=1)).ravel()
            bins_with_signal |= np.asarray(positive.sum(axis=0)).ravel() > 0

        cell_summaries.append(pd.DataFrame({
            "mark": adata.obs["mark"].to_numpy(),
            "total_counts": total_counts,
            "bins_with_signal": detected_bins,
        }, index=adata.obs_names))

        bin_summaries.append({
            "file": path.name,
            "total_bins": adata.n_vars,
            "bins_with_signal": bins_with_signal.sum(),
            "bins_without_signal": (~bins_with_signal).sum(),
            "without_signal_percent": 100 * (~bins_with_signal).mean(),
        })

        metadata.append(adata.obs[["mark", "day", "celltype_general"]].copy())
    finally:
        adata.file.close()

cell_stats = pd.concat(cell_summaries)
cell_metadata = pd.concat(metadata)

print("\n1. Counts and detected bins per cell: distribution by mark")
display(
    cell_stats.groupby("mark")[["total_counts", "bins_with_signal"]]
    .describe().T.round(1)
)

print("\n2. Bins without signal in any cell")
display(pd.DataFrame(bin_summaries).round(2))

print("\n3. Cell types per day and histone mark")
display(pd.crosstab(
    cell_metadata["celltype_general"],
    [cell_metadata["mark"], cell_metadata["day"]],
))